# 01. Generate Ours Static Weight

Generated clean workflow notebook for `ZexuZang/vggt-demo`.


## Purpose
Generate our motion-aware static weights for DROID-W from Bonn RGB-D Dynamic frames.

Core method:
VGGT geometry → Any4D scene flow → CoTracker point trajectories → point-level dynamic score → per-frame DROID-W static weights.

Run `00_environment_droidw_colab.ipynb` first after runtime reset.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content/vggt-demo
!git pull origin main


## 10. Bonn + DROID-W Quick Test

This section is for testing whether your VGGT + Any4D + CoTracker dynamic reliability can produce per-frame `static_weight` maps for DROID-W.

Important: for a real DROID-W experiment, VGGT / Any4D / CoTracker must run on the **same Bonn sequence frames** used by DROID-W. Do not use the old `any4d_baby` paths for this test.


In [ ]:
# ===== Clear RAM / GPU memory and reset Bonn path =====
%reset -f

import os
import gc
from pathlib import Path

try:
    import torch
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
except Exception as e:
    print("Torch cache clear skipped:", e)

gc.collect()

# Rebuild Bonn path after reset
BONN_SEQ_DIR = Path("/content/drive/MyDrive/Datasets/Bonn/person_tracking")
if not BONN_SEQ_DIR.exists():
    fallback = Path("/content/drive/MyDrive/Datasets/Bonn/rgbd_bonn_person_tracking")
    if fallback.exists():
        BONN_SEQ_DIR = fallback

RGB_DIR = BONN_SEQ_DIR / "rgb"

assert RGB_DIR.exists(), f"RGB folder not found: {RGB_DIR}"

os.environ["RGB_DIR"] = str(RGB_DIR)

print("Memory cleanup done.")
print("BONN_SEQ_DIR =", BONN_SEQ_DIR)
print("RGB_DIR =", RGB_DIR)

!nvidia-smi
!find "$RGB_DIR" -maxdepth 1 -type f | wc -l

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

SEQ_NAME = "person_tracking"

# Bonn sequence folder. Use the renamed folder first; otherwise fallback to the original unzip folder.
BONN_SEQ_DIR = Path("/content/drive/MyDrive/Datasets/Bonn/person_tracking")
if not BONN_SEQ_DIR.exists():
    fallback = Path("/content/drive/MyDrive/Datasets/Bonn/rgbd_bonn_person_tracking")
    if fallback.exists():
        BONN_SEQ_DIR = fallback

RGB_DIR = BONN_SEQ_DIR / "rgb"

print("BONN_SEQ_DIR:", BONN_SEQ_DIR)
print("RGB_DIR:", RGB_DIR)

assert RGB_DIR.exists(), f"RGB folder not found: {RGB_DIR}"

print("RGB examples:")
!find "$RGB_DIR" -maxdepth 1 -type f | head
!find "$RGB_DIR" -maxdepth 1 -type f | wc -l


### 10.1 Export VGGT predictions on Bonn frames

In [ ]:
## 路径修复
from pathlib import Path
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=False)

SEQ_NAME = "person_tracking"

BONN_SEQ_DIR = Path("/content/drive/MyDrive/Datasets/Bonn/person_tracking")

if not BONN_SEQ_DIR.exists():
    fallback = Path("/content/drive/MyDrive/Datasets/Bonn/rgbd_bonn_person_tracking")
    if fallback.exists():
        BONN_SEQ_DIR = fallback

RGB_DIR = BONN_SEQ_DIR / "rgb"

print("BONN_SEQ_DIR =", BONN_SEQ_DIR)
print("RGB_DIR =", RGB_DIR)
print("RGB_DIR exists:", RGB_DIR.exists())

assert RGB_DIR.exists(), f"RGB folder not found: {RGB_DIR}"

# 关键：把 Python 变量写入 shell 环境变量
os.environ["BONN_SEQ_DIR"] = str(BONN_SEQ_DIR)
os.environ["RGB_DIR"] = str(RGB_DIR)

print("Shell RGB_DIR =", os.environ["RGB_DIR"])

!ls -lah "$RGB_DIR" | head
!find "$RGB_DIR" -maxdepth 1 -type f | head
!find "$RGB_DIR" -maxdepth 1 -type f | wc -l

In [ ]:
%cd /content/vggt-demo

!PYTHONPATH=/content/vggt-demo:$PYTHONPATH python scripts/any4d_mask/export_vggt_predictions.py \
  --image_dir "$RGB_DIR" \
  --output_npz outputs/vggt_pred_bonn_person_8.npz \
  --max_frames 8

### 10.2 Export Any4D predictions on Bonn frames

In [ ]:
%cd /content/vggt-demo
!git fetch origin
!git checkout droidw-integration
!git pull origin droidw-integration

In [ ]:
%cd /content/vggt-demo

# 1. 查看当前状态
!git status

# 2. 丢掉 Colab 本地临时修改
!git reset --hard
!git clean -fd

# 3. 拉取远程分支
!git fetch origin

# 4. 切换到 droidw-integration 分支
!git checkout droidw-integration

# 5. 强制和远程 droidw-integration 保持一致
!git reset --hard origin/droidw-integration

# 6. 检查现在是不是最新 commit
!git log --oneline -3

# 7. 检查文件里是否已经没有 from scripts.demo_inference
!grep -n "load_any4d_demo_functions\|from scripts.demo_inference\|importlib.util" scripts/any4d_mask/export_any4d_predictions.py | head -20

In [ ]:
%cd /content/vggt-demo/third_party/Any4D

!PYTHONPATH=/content/vggt-demo/third_party/Any4D:$PYTHONPATH python /content/vggt-demo/scripts/any4d_mask/export_any4d_predictions.py \
  --any4d_root /content/vggt-demo/third_party/Any4D \
  --image_dir "$RGB_DIR" \
  --checkpoint_path /content/vggt-demo/third_party/Any4D/checkpoints/any4d_4v_combined.pth \
  --output_npz /content/vggt-demo/outputs/any4d_pred_bonn_person_8.npz \
  --ref_img_idx 0 \
  --start_idx 0 \
  --end_idx 8

### 10.3 Run point-level dynamic scoring on Bonn frames

In [ ]:
%cd /content/vggt-demo

!python scripts/any4d_mask/cotracker_any4d_dynamic_points.py \
  --image_dir "$RGB_DIR" \
  --any4d_npz outputs/any4d_pred_bonn_person_8.npz \
  --vggt_npz outputs/vggt_pred_bonn_person_8.npz \
  --output_dir outputs/ours_bonn_person_temporal8 \
  --max_frames 8 \
  --grid_size 45 \
  --resize_max_side 768 \
  --local_resample \
  --local_step 5 \
  --local_pad 65 \
  --local_link_radius 28 \
  --local_seed_thresh 0.45 \
  --local_motion_gate 0.08 \
  --local_depth_gate 0.35 \
  --local_max_queries 3500 \
  --point_dynamic_thresh 0.50 \
  --temporal_any4d_thresh 0.30 \
  --min_visible_ratio 0.30 \
  --use_bg_motion_residual \
  --bg_model affine \
  --bg_residual_reduce max \
  --bg_residual_score_thresh 0.30 \
  --bg_residual_px_thresh 0.8 \
  --use_temporal_or \
  --temporal_min_bg_frames 1 \
  --temporal_min_visible_frames 2 \
  --use_depth_cc \
  --cc_radius 10 \
  --cc_depth_gate 0.25 \
  --cc_score_gate 0.60 \
  --cc_candidate_score_thresh 0.20 \
  --cc_min_component_points 2 \
  --cc_min_dynamic_points 1 \
  --cc_min_dynamic_ratio 0.05 \
  --cc_min_mean_score 0.15 \
  --cc_propagate \
  --point_radius 1 \
  --point_vis_radius 3 \
  --blur_sigma 0 \
  --mask_thresh 0.30 \
  --combine_with_prior none


### 10.4 Visualize the Bonn point-level result

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image
import json

out = Path("/content/vggt-demo/outputs/ours_bonn_person_temporal8")

summary = out / "cotracker_any4d_summary.json"
if summary.exists():
    print(summary.read_text()[:2500])

for name in [
    "representative_points_overlay.png",
    "point_any4d_motion_score_overlay.png",
    "point_bg_residual_score_overlay.png",
    "point_vggt_depth_overlay.png",
    "cotracker_any4d_dynamic_overlay.png",
    "cotracker_any4d_static_ba_overlay.png",
]:
    p = out / name
    if p.exists():
        print("\n", name)
        display(Image.open(p))


## 11. Export per-frame static weights for DROID-W

This converts `cotracker_any4d_tracks.npz` into per-frame `static_weight_npy/*.npy`.

These maps are the files that DROID-W will read later.


In [ ]:
from pathlib import Path

OURS_POINT_DIR = Path("/content/vggt-demo/outputs/ours_bonn_person_temporal8")
TRACKS_NPZ = OURS_POINT_DIR / "cotracker_any4d_tracks.npz"

print(TRACKS_NPZ)
print(TRACKS_NPZ.exists())

In [ ]:
from pathlib import Path

SEQ_NAME = "person_tracking"
OURS_POINT_DIR = Path("/content/vggt-demo/outputs/ours_bonn_person_temporal8")
TRACKS_NPZ = OURS_POINT_DIR / "cotracker_any4d_tracks.npz"
DROIDW_WEIGHT_DIR = Path(f"/content/vggt-demo/outputs/droidw_ours_weights/{SEQ_NAME}")

print("TRACKS_NPZ:", TRACKS_NPZ)
print("DROIDW_WEIGHT_DIR:", DROIDW_WEIGHT_DIR)

assert TRACKS_NPZ.exists(), f"tracks npz not found: {TRACKS_NPZ}. Run cell 10.3 first."


In [ ]:
%cd /content/vggt-demo

# Make sure the exporter script exists.
# If it is missing, add scripts/droidw/export_droidw_static_weights.py in VSCode, push to GitHub, then run git pull.
!test -f scripts/droidw/export_droidw_static_weights.py && echo "Exporter script found." || (echo "Exporter script NOT found. Run git pull after adding it in VSCode." && false)

!python scripts/droidw/export_droidw_static_weights.py \
  --tracks_npz "$TRACKS_NPZ" \
  --image_dir "$RGB_DIR" \
  --output_dir "$DROIDW_WEIGHT_DIR" \
  --point_radius 3 \
  --blur_sigma 1.5 \
  --visibility_thresh 0.5 \
  --min_static_weight 0.05 \
  --use_sequential_names


In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image
import json

out = Path(DROIDW_WEIGHT_DIR)
print("Output:", out)

summary_path = out / "summary.json"
if summary_path.exists():
    print(summary_path.read_text()[:2000])

for sub in ["dynamic_prob_png", "static_weight_png", "overlay_dynamic"]:
    print("\n==", sub, "==")
    imgs = sorted((out / sub).glob("*.png"))[:3]
    for p in imgs:
        print(p.name)
        display(Image.open(p))
